In [1]:
%load_ext dotenv
%dotenv

In [2]:
import pandas as pd
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer
import hashlib
import pdfplumber
from pathlib import Path
import re
from openai import OpenAI
import time

In [3]:
pc = Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), environment = os.environ.get('PINECONE_ENV'))

In [4]:
PDF_FOLDER = "CoinCleaning"  
CHUNK_SIZE = 500 
CHUNK_OVERLAP = 50  

In [5]:
def parse_filename_metadata(filepath: str) -> dict:
    """
    Parses metadata from filename formatted as:
        <DocName>_<ShortDescription>_<AuthorName>.pdf
 
    Returns a dict with keys: doc_name, description, author.
    Falls back gracefully if the filename doesn't match the convention.
    """
    stem = Path(filepath).stem          # filename without extension
    parts = stem.split("_", maxsplit=3) # split on first 3 underscores only
 
    if len(parts) == 4:
        doc_name, metal, description, author = parts
    else:
        raise ValueError("Missing Information")
 
    # Convert CamelCase or underscored tokens to readable strings
    doc_name_readable    = re.sub(r'(?<!^)(?=[A-Z])', ' ', doc_name).strip()
    metal_readable       = re.sub(r'(?<!^)(?=[A-Z])', ' ', metal).strip()
    description_readable = re.sub(r'(?<!^)(?=[A-Z])', ' ', description).strip()
    author_readable      = re.sub(r'(?<!^)(?=[A-Z])', ' ', author).strip()
 
    return {
        "doc_name":    doc_name_readable,
        "metal":       metal_readable,
        "description": description_readable,
        "author":      author_readable,
        "source_file": Path(filepath).name,
    }

In [6]:
def classify_chunk_metal(text: str, doc_level_metal: str, openai_client) -> str:
    """
    For mixed documents, uses a cheap LLM call to classify each chunk.
    For single-metal documents, just returns the doc-level metal directly.
    """
    # If the document is known to be single-metal, trust it
    if doc_level_metal.lower() in ("bronze", "silver"):
        return doc_level_metal.lower()

    response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    """You classify coin conservation text by the metal it applies to. 
                    Reply with exactly one word: bronze, silver, both, or general. 
                    Mechanically abbrasive techniques involving hard tools are bronze only. Strong chemical treatments are for silver only. Acetone and distilled water are general tools. Picks, scalpels, brass bristle brushes, etc are for bronze coins. Erasers, soft fiberglass brushes, wooden toothpicks are safe for both. Gringgott's chemicals, unless explicitly for silver, are for bronze coins. Waxing is applicable to both. Silvering and silvered are terms that apply to bronze coins with a silver plating, not silver coins. Silvered coins are bronze. Use 'general' only if the passage truly applies equally to all metals with no metal-specific advice but lean towards 'bronze' if unclear. Use 'both' if the passage explicitly discusses bronze AND silver with distinct advice for each."""
                )
            },
            {
                "role": "user",
                "content": text[:800]  # first 800 chars is usually enough to classify
            }
        ]
    )
    label = response.choices[0].message.content.strip().lower()
    return label if label in ("bronze", "silver", "bronze silver", "general") else "general"

In [7]:
def extract_text_from_pdf(filepath: str) -> str:
    """
    Extracts all text from a PDF using pdfplumber.
    Joins pages with double newlines to preserve paragraph boundaries.
    Falls back to pypdf if pdfplumber yields nothing.
    """
    full_text = []
 
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text.append(page_text.strip())
 
    if not full_text:
        # Fallback: try pypdf
        from pypdf import PdfReader
        reader = PdfReader(filepath)
        for page in reader.pages:
            t = page.extract_text()
            if t:
                full_text.append(t.strip())
 
    return "\n\n".join(full_text)

In [8]:
def clean_text(text: str) -> str:
    """
    Light cleaning to fix common PDF extraction artifacts:
    - Collapse excessive whitespace within lines
    - Normalize line endings
    - Remove lines that are purely page numbers or headers/footers (heuristic)
    """
    lines = text.splitlines()
    cleaned = []
 
    for line in lines:
        line = line.strip()
        # Drop very short lines that are likely page numbers or artefacts
        if len(line) <= 3 and re.match(r'^[\d\s\-–—]*$', line):
            continue
        # Collapse internal whitespace
        line = re.sub(r'[ \t]+', ' ', line)
        cleaned.append(line)
 
    return "\n".join(cleaned)

In [9]:
def split_into_paragraphs(text: str) -> list[str]:
    """
    Splits text on blank lines (paragraph boundaries), which pdfplumber
    typically preserves when a document has clear paragraph structure.
    """
    # Split on one or more blank lines
    raw_paragraphs = re.split(r'\n{2,}', text)
    # Filter out empty/whitespace-only segments
    return [p.strip() for p in raw_paragraphs if p.strip()]

In [10]:
def chunk_text(text: str, doc_metal: str, openai_client,
               chunk_size: int = CHUNK_SIZE, 
               overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """
    Chunks text in a paragraph-aware way:
      1. Splits by paragraph boundaries first.
      2. Accumulates paragraphs into chunks up to `chunk_size` words.
      3. Adds `overlap` words of the previous chunk to the start of the next
         for context continuity (important for RAG retrieval quality).
      4. Classifies each chunk by metal type and prepends the label to the text.

    Returns a list of dicts with keys: text, metal_label
    """
    paragraphs = split_into_paragraphs(text)
    raw_chunks = []
    current_words = []

    for para in paragraphs:
        para_words = para.split()

        if current_words and len(current_words) + len(para_words) > chunk_size:
            raw_chunks.append(" ".join(current_words))
            current_words = current_words[-overlap:] if overlap > 0 else []

        current_words.extend(para_words)

        while len(current_words) > chunk_size:
            raw_chunks.append(" ".join(current_words[:chunk_size]))
            current_words = current_words[chunk_size - overlap:]

    if current_words:
        raw_chunks.append(" ".join(current_words))

    # Classify and annotate each chunk
    chunk_dicts = []
    for i, raw_text in enumerate(raw_chunks):
        print(f"    Classifying chunk {i + 1}/{len(raw_chunks)}...")
        metal_label = classify_chunk_metal(raw_text, doc_metal, openai_client)
        chunk_dicts.append({
            "text":        f"[Applies to: {metal_label}]\n{raw_text}",
            "metal_label": metal_label,
        })

    return chunk_dicts

In [11]:
def process_pdf_folder(folder: str, openai_client) -> list[dict]:
    """
    Processes all PDFs in `folder`.
    Returns a flat list of chunk dicts, each shaped like:
 
    {
        "doc_name":    "Electrochemical Cleaning",
        "description": "Electrolysis Methods For Coins",
        "author":      "John Smith",
        "source_file": "ElectrochemicalCleaning_ElectrolysisMethodsForCoins_JohnSmith.pdf",
        "chunk_index": 3,           # 0-based position within this document
        "total_chunks": 12,         # total chunks from this document
        "text":        "..."        # the actual chunk text for embedding
    }
    """
    all_chunks = []
    pdf_paths = sorted(Path(folder).glob("*.pdf"))
 
    if not pdf_paths:
        print(f"No PDFs found in '{folder}'. Check the folder path.")
        return all_chunks
 
    for pdf_path in pdf_paths:
        print(f"\nProcessing: {pdf_path.name}")
 
        # 1. Parse metadata from filename
        metadata = parse_filename_metadata(str(pdf_path))
        print(f"  → Doc: {metadata['doc_name']} | Author: {metadata['author']}")
 
        # 2. Extract raw text
        raw_text = extract_text_from_pdf(str(pdf_path))
        if not raw_text.strip():
            print(f"  ⚠ No text extracted — PDF may be scanned. Consider OCR.")
            continue
 
        # 3. Clean the text
        cleaned = clean_text(raw_text)
 
        # 4. Chunk the text and classify by metal
        chunks = chunk_text(cleaned, doc_metal=metadata["metal"], openai_client=openai_client)
        print(f"  → {len(chunks)} chunks produced")

        # 5. Build chunk dicts
        for i, chunk in enumerate(chunks):
            record = {
                **metadata,
                "chunk_index":  i,
                "total_chunks": len(chunks),
                "text":         chunk["text"],        # includes [Applies to: ...] prefix
                "metal_label":  chunk["metal_label"], # stored as Pinecone metadata field
            }
            all_chunks.append(record)
 
    print(f"\n✓ Total chunks across all documents: {len(all_chunks)}")
    return all_chunks

In [13]:
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

chunks = process_pdf_folder(PDF_FOLDER, openai_client)


Processing: AMethodForCleaningAncientBronzeCoins_Bronze_ExpertCleaner_SaulRoll.pdf
  → Doc: A Method For Cleaning Ancient Bronze Coins | Author: Saul Roll
    Classifying chunk 1/27...
    Classifying chunk 2/27...
    Classifying chunk 3/27...
    Classifying chunk 4/27...
    Classifying chunk 5/27...
    Classifying chunk 6/27...
    Classifying chunk 7/27...
    Classifying chunk 8/27...
    Classifying chunk 9/27...
    Classifying chunk 10/27...
    Classifying chunk 11/27...
    Classifying chunk 12/27...
    Classifying chunk 13/27...
    Classifying chunk 14/27...
    Classifying chunk 15/27...
    Classifying chunk 16/27...
    Classifying chunk 17/27...
    Classifying chunk 18/27...
    Classifying chunk 19/27...
    Classifying chunk 20/27...
    Classifying chunk 21/27...
    Classifying chunk 22/27...
    Classifying chunk 23/27...
    Classifying chunk 24/27...
    Classifying chunk 25/27...
    Classifying chunk 26/27...
    Classifying chunk 27/27...
  → 27 chunks pr

In [14]:
for record in chunks[:2]:
    print("\n── CHUNK PREVIEW ──────────────────────────────────────────────")
    for key, val in record.items():
        if key == "text":
            print(f"  text (first 300 chars): {val[:300]}...")
        else:
            print(f"  {key}: {val}")


── CHUNK PREVIEW ──────────────────────────────────────────────
  doc_name: A Method For Cleaning Ancient Bronze Coins
  metal: Bronze
  description: Expert Cleaner
  author: Saul Roll
  source_file: AMethodForCleaningAncientBronzeCoins_Bronze_ExpertCleaner_SaulRoll.pdf
  chunk_index: 0
  total_chunks: 27
  text (first 300 chars): [Applies to: bronze]
A Method for Cleaning Ancient Bronze Coins With a long introduction, and much annoying detail Plus some commentaries on bronze disease To which is added a Bitter Afterword And a Bitchy Epilogue, followed by an Appendix of coin photographs Saúl Roll www.romanorum.com© BIG SMALL P...
  metal_label: bronze

── CHUNK PREVIEW ──────────────────────────────────────────────
  doc_name: A Method For Cleaning Ancient Bronze Coins
  metal: Bronze
  description: Expert Cleaner
  author: Saul Roll
  source_file: AMethodForCleaningAncientBronzeCoins_Bronze_ExpertCleaner_SaulRoll.pdf
  chunk_index: 1
  total_chunks: 27
  text (first 300 chars): [Appli

In [15]:
pc = Pinecone(api_key = os.environ.get('PINECONE_API_KEY'), environment = os.environ.get('PINECONE_ENV'))

In [16]:
INDEX_NAME    = "ancient-coin-cleaning"
EMBEDDING_DIM = 1536
METRIC        = "cosine"

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME in existing_indexes:
    pc.delete_index(INDEX_NAME)
    print(f"🗑 Existing index '{INDEX_NAME}' deleted.")
    # Deletion is async — poll until it's fully gone before recreating
    while INDEX_NAME in [idx.name for idx in pc.list_indexes()]:
        print("  Waiting for deletion to complete...")
        time.sleep(2)

pc.create_index(
    name=INDEX_NAME,
    dimension=EMBEDDING_DIM,
    metric=METRIC,
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)
print(f"✓ Index '{INDEX_NAME}' created.")

# Creation is also async — poll until ready before connecting
while not pc.describe_index(INDEX_NAME).status["ready"]:
    print("  Waiting for index to be ready...")
    time.sleep(2)

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

🗑 Existing index 'ancient-coin-cleaning' deleted.
✓ Index 'ancient-coin-cleaning' created.
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Mar 2026 16:48:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '42',
                                    'x-pinecone-request-latency-ms': '41',
                                    'x-pinecone-response-duration-ms': '43'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [17]:

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 100  # OpenAI allows up to 2048 inputs per request; 100 is safe

def embed_chunks(chunks: list[dict], model: str = EMBEDDING_MODEL) -> list[dict]:
    """
    Adds an 'embedding' key to each chunk dict.
    Processes in batches to stay within API rate limits.
    """
    texts = [c["text"] for c in chunks]
    all_embeddings = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        response = openai_client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} chunks...")
        time.sleep(0.25)  # gentle rate-limit buffer

    for chunk, embedding in zip(chunks, all_embeddings):
        chunk["embedding"] = embedding

    print(f"✓ All {len(chunks)} chunks embedded.")
    return chunks

chunks = embed_chunks(chunks)


  Embedded 100/591 chunks...
  Embedded 200/591 chunks...
  Embedded 300/591 chunks...
  Embedded 400/591 chunks...
  Embedded 500/591 chunks...
  Embedded 591/591 chunks...
✓ All 591 chunks embedded.


In [18]:
import hashlib

UPSERT_BATCH_SIZE = 100  # Pinecone recommends 100 vectors per upsert batch

def make_vector_id(chunk: dict) -> str:
    """
    Creates a stable, unique ID from source file + chunk index.
    Using a hash avoids issues with special characters in filenames.
    """
    raw = f"{chunk['source_file']}__chunk_{chunk['chunk_index']}"
    return hashlib.md5(raw.encode()).hexdigest()

def upsert_chunks(index, chunks: list[dict], batch_size: int = UPSERT_BATCH_SIZE):
    """
    Upserts all embedded chunks to Pinecone.
    Metadata fields (everything except the raw embedding) are stored
    alongside the vector for filtered retrieval later.
    """
    vectors = []
    for chunk in chunks:
        vector_id = make_vector_id(chunk)
        metadata = {
            "doc_name":     chunk["doc_name"],
            "description":  chunk["description"],
            "author":       chunk["author"],
            "source_file":  chunk["source_file"],
            "chunk_index":  chunk["chunk_index"],
            "total_chunks": chunk["total_chunks"],
            "text":         chunk["text"],   # stored for retrieval — returned with matches
        }
        vectors.append({
            "id":       vector_id,
            "values":   chunk["embedding"],
            "metadata": metadata,
        })

    # Upsert in batches
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)
        print(f"  Upserted {min(i + batch_size, len(vectors))}/{len(vectors)} vectors...")

    print(f"✓ All {len(vectors)} vectors upserted to '{INDEX_NAME}'.")

upsert_chunks(index, chunks)

# Confirm final state
print("\nIndex stats after upsert:")
print(index.describe_index_stats())

  Upserted 100/591 vectors...
  Upserted 200/591 vectors...
  Upserted 300/591 vectors...
  Upserted 400/591 vectors...
  Upserted 500/591 vectors...
  Upserted 591/591 vectors...
✓ All 591 vectors upserted to 'ancient-coin-cleaning'.

Index stats after upsert:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Mar 2026 16:48:59 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '39',
                                    'x-pinecone-request-latency-ms': '38',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'ve